In [ ]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib as mpl
%matplotlib inline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from graphviz import Source
from sklearn.tree import export_graphviz

!pip install graphviz

In [ ]:
trainData = pd.read_csv("train.csv")
testData = pd.read_csv("test.csv") 

In [ ]:
#merge two datasets
titanic = pd.concat([trainData, testData], sort=True)

In [ ]:
titanic.head()

In [ ]:
titanic.info()
# pay attention to object data types

In [ ]:
sns.heatmap(titanic.isnull(), cbar=False)
# the figure shows the null values.

In [ ]:
# Drop Cabin
titanic.drop('Cabin',axis=1, inplace=True)
# Drop Name
titanic.drop('Name',axis=1, inplace=True)
# Drop ticket
titanic.drop('Ticket',axis=1, inplace=True)
# drop passenger id
titanic.drop('PassengerId',axis=1, inplace=True)



In [ ]:
#remove rows that has null values for fare and embarked
titanic.dropna(subset=['Fare','Embarked'],inplace=True)


In [ ]:
# convert embark to category
titanic['Embarked'] = titanic['Embarked'].astype('category')
#convert sext to category
titanic['Sex'] = titanic['Sex'].astype('category')
#convert Pclass to category
titanic['Pclass'] = titanic['Pclass'].astype('category')


In [ ]:
sns.heatmap(titanic.isnull(), cbar=False)
#data frame has null values only for survived and age.

In [ ]:
titanic.info()
#all values are numeric except embarked,pclass and sex

In [ ]:
#convert categorical data to numerical
df_Embarked= pd.get_dummies(titanic['Embarked'],'embark')
df_Pclass = pd.get_dummies(titanic['Pclass'],'pclass')
df_sex = pd.get_dummies(titanic['Sex'],'sex')

#Concat new columns to original dataframe 
titanic = pd.concat([titanic, df_Embarked, df_Pclass,df_sex], axis=1)

In [ ]:
# drop original categorical variables
titanic.drop(columns=['Embarked','Pclass','Sex'], axis=1,inplace=True)

In [ ]:
titanic.head()

In [ ]:
#split the data into two sets, one with age, one without age
#we will use data_with_age to build the model
data_age_missing = titanic[titanic['Age'].isnull()]
data_with_age = titanic[titanic['Age'].notnull()]

In [ ]:
#build training and testing dataset

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(data_with_age.drop(columns=['Age','Survived'],axis=1), data_with_age['Age'], test_size=0.2, random_state=42)

In [ ]:
#build a decision tree regressor model
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(X_train, y_train)

In [ ]:
# get rmse
from sklearn.metrics import mean_squared_error

age_predictions = tree_reg.predict(X_train)
tree_mse = mean_squared_error(y_train, age_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

In [ ]:
# mae(mean absolute error) is also another metric
from sklearn.metrics import mean_absolute_error

lin_mae = mean_absolute_error(y_train, age_predictions)
lin_mae

In [ ]:
#cross validation
from sklearn.model_selection import cross_val_score

scores = cross_val_score(tree_reg, X_train, y_train,
                         scoring="neg_mean_squared_error", cv=5)
tree_rmse_scores = np.sqrt(-scores)
tree_rmse_scores

# RMSE scores for cross-validation differ a lot from that of original model. This means we have overfitting model.

In [ ]:
# the model is overfitting!!!

In [ ]:
# We regularize model by setting some hyperparameters. 

from sklearn.model_selection import GridSearchCV

param_grid = [
    # try 16 (4?4) combinations of hyperparameters
    {'min_samples_leaf': [3, 10, 30,50], 'max_features': [2, 4, 6, 8]}
  ]

# train across 5 folds, that's a total of (16)*5=80 rounds of training 
grid_search = GridSearchCV(tree_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(X_train, y_train)

In [ ]:
#get best parameters from gridsearchcv
grid_search.best_params_

In [ ]:
#build another model with the best parameters from gridsearchcv
from sklearn.tree import DecisionTreeRegressor

tree_reg = DecisionTreeRegressor(random_state=42, max_features=6, min_samples_leaf=30)
tree_reg.fit(X_train, y_train)

In [ ]:
#get rmse for new model
age_predictions = tree_reg.predict(X_train)
tree_mse = mean_squared_error(y_train, age_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

In [ ]:
#get rmse for cross-validation models
scores = cross_val_score(tree_reg, X_train, y_train,
                         scoring="neg_mean_squared_error", cv=5)
tree_rmse_scores = np.sqrt(-scores)
tree_rmse_scores

In [ ]:
# rmse values are very similar. We no longer have overfitting issue.

In [ ]:
# make predictions for null values in data_age_missing data set
Age_predictions_for_missing_data = tree_reg.predict(data_age_missing.drop(columns=['Age','Survived'],axis=1))

In [ ]:
#build a dataframe containing predictions. 
#please pay attention to index. It must be data_age_missing's index info so that have matching predictions.
predicted_age_df = pd.DataFrame(Age_predictions_for_missing_data, index=data_age_missing.index, columns=['Age'])


In [ ]:
# put predictions back into data_age-missing
data_age_missing = pd.concat([predicted_age_df, data_age_missing.drop(['Age'],axis=1)], axis=1)
data_age_missing

In [ ]:
# merge two datasets
titanic = pd.concat([data_age_missing,data_with_age], sort=True)


In [ ]:
titanic.info()

In [ ]:
sns.heatmap(titanic.isnull(), cbar=False)

In [ ]:
###

In [ ]:
results = pd.DataFrame({
    'actual': y_test,
    'prediction': predictions
})
results